# 02 — Building the sales intelligence agent

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/02_sales_intelligence/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb) — Pipeline architecture and data contracts
- Familiarity with OpenAI API and prompt engineering

**What you will learn**:
- How to build a synthetic prospect database with realistic signals
- How to implement a research agent that extracts personalization hooks
- How to generate deeply personalized outreach emails via LLM
- How to classify prospect responses and handle objections automatically
- How to wire every component into an end-to-end sales pipeline

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0" "numpy>=1.24.0"

import os
import json
import random
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

random.seed(42)
np.random.seed(42)
print("Setup complete.")

## Section 1 — Prospect database

A realistic prospect database is the fuel for every downstream component.
Each record should include firmographic data *and* behavioural signals —
recent news, job postings, and technology choices — because those signals
are what let us write emails that feel hand-researched.

We build a synthetic database of 20 companies spanning four industries so
we can test the pipeline without external data sources.

In [ ]:
INDUSTRIES = ["FinTech", "HealthTech", "SaaS / DevTools", "E-Commerce"]
FUNDING_STAGES = ["Seed", "Series A", "Series B", "Series C", "Series D+"]
TECH_STACKS = {
    "FinTech":         [["Python", "Kafka", "PostgreSQL", "React"],
                        ["Java", "Spark", "MongoDB", "Angular"]],
    "HealthTech":      [["Python", "FHIR", "PostgreSQL", "React"],
                        ["TypeScript", "AWS Lambda", "DynamoDB", "Next.js"]],
    "SaaS / DevTools": [["Go", "Kubernetes", "Redis", "Vue"],
                        ["Rust", "gRPC", "ClickHouse", "Svelte"]],
    "E-Commerce":      [["Python", "Shopify API", "PostgreSQL", "Next.js"],
                        ["TypeScript", "Stripe", "Firebase", "React"]],
}

NEWS_TEMPLATES = [
    "Raised {amount} in {stage} funding led by {investor}",
    "Launched new {product} product line targeting {market}",
    "Appointed {name} as new CTO from {origin}",
    "Expanded into {region} with new office in {city}",
    "Published engineering blog on migrating to {tech}",
]

JOB_TEMPLATES = [
    "Hiring {n} senior backend engineers",
    "Open role: VP of Engineering",
    "Recruiting ML engineers for {team} team",
    "Looking for DevOps lead to scale infrastructure",
    "Posted 3 openings in data engineering",
]

def _random_news(industry: str) -> list[str]:
    fillers = {
        "amount": random.choice(["$12M", "$28M", "$55M", "$110M"]),
        "stage": random.choice(FUNDING_STAGES),
        "investor": random.choice(["Sequoia", "a16z", "Accel", "Founders Fund"]),
        "product": random.choice(["analytics", "compliance", "integration", "AI assistant"]),
        "market": random.choice(["mid-market", "enterprise", "SMB"]),
        "name": random.choice(["Jordan Lee", "Priya Patel", "Marcus Chen"]),
        "origin": random.choice(["Google", "Meta", "Stripe", "Datadog"]),
        "region": random.choice(["EMEA", "APAC", "LATAM"]),
        "city": random.choice(["London", "Singapore", "São Paulo", "Berlin"]),
        "tech": random.choice(["Kubernetes", "Rust", "event-driven arch"]),
    }
    k = random.randint(1, 3)
    return [t.format(**fillers) for t in random.sample(NEWS_TEMPLATES, k)]

def _random_jobs() -> list[str]:
    fillers = {"n": random.randint(3, 12), "team": random.choice(["search", "recommendations", "platform"])}
    k = random.randint(1, 2)
    return [t.format(**fillers) for t in random.sample(JOB_TEMPLATES, k)]

COMPANY_NAMES = [
    "NovaPay", "MedSync", "CodeLoom", "CartVibe", "LedgerAI",
    "PulseHealth", "ShipStack", "BuyBetter", "TrustVault", "CareLink",
    "DevHarbor", "StyleLoop", "PayGrid", "GenoWell", "BuildKite",
    "DealSprint", "HealthBridge", "PipeForge", "ShopNova", "CompliBot",
]

prospects: list[dict] = []
for i, name in enumerate(COMPANY_NAMES):
    industry = INDUSTRIES[i % len(INDUSTRIES)]
    prospects.append({
        "name": name,
        "industry": industry,
        "size": random.choice([50, 120, 250, 500, 1000, 2000]),
        "recent_news": _random_news(industry),
        "job_postings": _random_jobs(),
        "tech_stack": random.choice(TECH_STACKS[industry]),
        "funding_stage": random.choice(FUNDING_STAGES),
        "contact_name": random.choice(["Alex Rivera", "Sam Patel", "Jordan Kim",
                                        "Taylor Nguyen", "Morgan Chen"]),
        "contact_title": random.choice(["VP Engineering", "CTO", "Head of Platform",
                                         "Director of Sales", "CEO"]),
    })

df_prospects = pd.DataFrame(prospects)
print(f"Prospect database: {len(prospects)} companies across {df_prospects['industry'].nunique()} industries\n")
print(df_prospects[["name", "industry", "size", "funding_stage"]].to_string(index=False))

In [ ]:
# Quick-look: distribution of prospects by industry and funding stage
print("Prospects by industry:")
print(df_prospects["industry"].value_counts().to_string())
print(f"\nProspects by funding stage:")
print(df_prospects["funding_stage"].value_counts().to_string())
print(f"\nCompany size — mean: {df_prospects['size'].mean():.0f}, "
      f"median: {df_prospects['size'].median():.0f}")
print(f"\nSample prospect record:")
print(json.dumps(prospects[0], indent=2))

## Section 2 — Research agent

The research agent is the most important component in the pipeline.
It takes raw prospect data and produces a structured *research brief*
containing 3-5 **personalization hooks** — concrete facts about the
prospect that a salesperson could reference in an email.

Hooks fall into three depth tiers:

| Tier | Example | Typical reply-rate lift |
|------|---------|------------------------|
| Surface | Uses company name / title | +1 % |
| Contextual | References recent funding round | +5 % |
| Deep | Infers pain from job postings + tech stack | +12 % |

In [ ]:
RESEARCH_SYSTEM_PROMPT = """You are a B2B sales research analyst. Given raw data about
a prospect company, produce a structured research brief.

Output JSON with these keys:
- company_summary: 2-sentence overview
- pain_signals: list of 2-4 likely pain points inferred from the data
- personalization_hooks: list of 3-5 specific facts an SDR can reference
- recommended_angle: the single strongest opening angle for outreach
- confidence: float 0-1 indicating data quality

Be specific. Never fabricate facts not supported by the input data."""

def research_prospect(prospect: dict) -> dict:
    """Run the research agent on a single prospect and return a brief."""
    user_msg = json.dumps(prospect, indent=2)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RESEARCH_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format={"type": "json_object"},
        temperature=0.4,
    )
    return json.loads(response.choices[0].message.content)

# Demo on first 3 prospects
for p in prospects[:3]:
    brief = research_prospect(p)
    print(f"\n{'=' * 60}")
    print(f"Company : {p['name']} ({p['industry']})")
    print(f"Summary : {brief.get('company_summary', 'N/A')}")
    print(f"Hooks   : {len(brief.get('personalization_hooks', []))}")
    for h in brief.get("personalization_hooks", []):
        print(f"  \u2022 {h}")
    print(f"Angle   : {brief.get('recommended_angle', 'N/A')}")
    print(f"Confidence: {brief.get('confidence', 0):.2f}")

## Section 3 — Personalized email generator

With the research brief in hand we can generate emails that feel
hand-written. The generator takes three inputs:

1. **Research brief** — the hooks and pain signals from Section 2
2. **ICP definition** — our ideal customer profile (industry, size, role)
3. **Value proposition** — what our product does for the prospect

We compare a *generic template* with an *AI-personalized* version for
the same prospect to make the quality gap viscerally obvious.

In [ ]:
ICP = {
    "target_industries": ["FinTech", "SaaS / DevTools", "HealthTech", "E-Commerce"],
    "company_size_range": "50-2000 employees",
    "target_titles": ["VP Engineering", "CTO", "Head of Platform"],
    "product": "DevPipe — AI-powered CI/CD platform",
    "value_prop": "Cut deployment cycle time by 60% and reduce pipeline failures by 45%",
}

GENERIC_TEMPLATE = """Hi {contact_name},

I'm reaching out because {product} helps companies like {company}
improve their engineering workflow. We've helped teams cut deployment
times significantly.

Would you be open to a 15-minute call this week?

Best,
Alex Rivera"""

EMAIL_SYSTEM_PROMPT = """You are an expert B2B sales copywriter. Write a short,
personalized cold email (under 150 words) using the research brief and value
proposition provided. Rules:
- Use at least 2 personalization hooks from the brief
- Reference a specific pain signal
- Include a clear, low-friction CTA
- Professional but conversational tone
- No generic flattery

Output JSON with keys: subject, body, hooks_used (list of strings)."""

def generate_email(prospect: dict, brief: dict, icp: dict) -> dict:
    """Generate a personalized email for a prospect given their research brief."""
    user_msg = json.dumps({"prospect": prospect, "brief": brief, "icp": icp}, indent=2)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": EMAIL_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    return json.loads(response.choices[0].message.content)

# Side-by-side comparison for 3 prospects
for p in prospects[:3]:
    brief = research_prospect(p)
    ai_email = generate_email(p, brief, ICP)
    generic = GENERIC_TEMPLATE.format(
        contact_name=p["contact_name"],
        product=ICP["product"],
        company=p["name"],
    )
    print(f"\n{'=' * 60}")
    print(f"Prospect: {p['name']} \u2014 {p['contact_name']} ({p['contact_title']})")
    print(f"\n--- GENERIC TEMPLATE ---\n{generic}")
    print(f"\n--- AI-PERSONALIZED ---")
    print(f"Subject: {ai_email.get('subject', '')}")
    print(ai_email.get("body", ""))
    print(f"\nHooks used: {ai_email.get('hooks_used', [])}")

## Section 4 — Response classifier

Once emails are sent we need to classify every response into an
actionable category so the right follow-up fires automatically.

| Category | Example reply | Action |
|----------|--------------|--------|
| `positive` | "Sure, let's find a time" | Book meeting |
| `objection_price` | "Too expensive for us right now" | ROI follow-up |
| `objection_timing` | "Not a priority this quarter" | Nurture sequence |
| `objection_authority` | "I'm not the right person" | Ask for referral |
| `not_interested` | "Please remove me from your list" | Suppress |
| `out_of_office` | "I'm OOO until March 10" | Reschedule send |
| `bounce` | "Address not found" | Mark invalid |

We use OpenAI few-shot classification for high accuracy across all
categories.

In [ ]:
RESPONSE_CATEGORIES = [
    "positive", "objection_price", "objection_timing",
    "objection_authority", "not_interested", "out_of_office", "bounce",
]

FEW_SHOT_EXAMPLES = [
    ("Sure, I'd love to chat. How about Thursday?", "positive"),
    ("Sounds interesting — can you send a calendar link?", "positive"),
    ("This is way out of our budget right now.", "objection_price"),
    ("We already have a solution and it's cheaper.", "objection_price"),
    ("We're in a code freeze until Q2, bad timing.", "objection_timing"),
    ("Maybe reach back out in 6 months?", "objection_timing"),
    ("I'm not the decision maker — try our VP Eng.", "objection_authority"),
    ("You'd need to talk to our CTO about this.", "objection_authority"),
    ("Not interested, please don't email me again.", "not_interested"),
    ("Remove me from your list.", "not_interested"),
    ("I'm out of office until March 15. Back on the 16th.", "out_of_office"),
    ("OOO — limited access to email this week.", "out_of_office"),
    ("Delivery failed: address not found.", "bounce"),
    ("550 User unknown.", "bounce"),
]

CLASSIFY_SYSTEM = """You are a sales response classifier. Classify the prospect's
reply into exactly one category. Respond with JSON: {"category": "...", "confidence": 0.0-1.0}

Categories: positive, objection_price, objection_timing, objection_authority,
not_interested, out_of_office, bounce."""

def classify_response(reply_text: str) -> dict:
    """Classify a prospect reply into one of the predefined categories."""
    few_shot_msgs: list[dict] = []
    for text, cat in FEW_SHOT_EXAMPLES:
        few_shot_msgs.append({"role": "user", "content": text})
        few_shot_msgs.append({"role": "assistant",
                              "content": json.dumps({"category": cat, "confidence": 0.95})})

    messages = [{"role": "system", "content": CLASSIFY_SYSTEM}] + few_shot_msgs + [
        {"role": "user", "content": reply_text}
    ]
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        response_format={"type": "json_object"}, temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)

# Test on 15 sample responses
TEST_REPLIES = [
    "Let's do it — how about next Tuesday at 2 PM?",
    "Interesting, but our contract with Acme runs through December.",
    "Way too pricey for a startup our size.",
    "I'm on PTO until April 1, ping me then.",
    "This isn't relevant to what we do. Please stop.",
    "Forwarding to our Head of Infra, she'd be the right contact.",
    "550 5.1.1 The email account does not exist.",
    "We just signed a 2-year deal with a competitor, bad timing.",
    "Sounds great — send over some case studies and let's talk.",
    "You'd need to loop in our CTO, I handle marketing.",
    "We're focused on other priorities this quarter.",
    "Can you send pricing details? We might have budget in Q3.",
    "Please remove me from all future communications.",
    "Hi, I'm out of the office with limited email access.",
    "This could be useful — are you free for a 15 min call Friday?",
]

print(f"{'Reply (truncated)':<50} {'Category':<25} {'Conf':>5}")
print("-" * 82)
for reply in TEST_REPLIES:
    result = classify_response(reply)
    cat = result.get("category", "unknown")
    conf = result.get("confidence", 0.0)
    print(f"{reply[:48]:<50} {cat:<25} {conf:>5.2f}")

## Section 5 — Objection handling agent

Not every objection is a "no" — many are invitations to reframe.
The objection handler selects a strategy based on the classification
and generates an appropriate follow-up.

| Objection type | Strategy | Goal |
|---------------|----------|------|
| Price | ROI comparison + case study | Reframe cost as investment |
| Timing | Nurture with value content | Stay top-of-mind |
| Authority | Polite referral request | Reach the decision maker |

In [ ]:
OBJECTION_STRATEGIES = {
    "objection_price": (
        "Acknowledge the budget concern, then reframe with a concrete ROI example. "
        "Mention a customer of similar size who saw measurable savings."
    ),
    "objection_timing": (
        "Respect their timeline. Offer a no-commitment resource (case study, report) "
        "and suggest a specific follow-up date."
    ),
    "objection_authority": (
        "Thank them, ask who the right person is, and request a warm introduction. "
        "Keep it short and easy to forward."
    ),
}

FOLLOWUP_SYSTEM = """You are an expert B2B sales rep writing a follow-up email
after receiving an objection. Use the provided strategy. Keep the email under
100 words, professional but warm. Output JSON: {"subject": "...", "body": "..."}."""

def generate_followup(
    prospect: dict,
    original_reply: str,
    objection_type: str,
) -> dict:
    """Generate a follow-up email tailored to the objection type."""
    strategy = OBJECTION_STRATEGIES.get(objection_type, "Respond politely.")
    user_msg = json.dumps({
        "prospect": prospect,
        "their_reply": original_reply,
        "objection_type": objection_type,
        "strategy": strategy,
    }, indent=2)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": FOLLOWUP_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    return json.loads(response.choices[0].message.content)

# Demo 3 objection-response pairs
OBJECTION_DEMOS = [
    (prospects[0], "We just don't have the budget for new tools right now.", "objection_price"),
    (prospects[1], "Interesting, but we're in the middle of a platform migration — maybe Q3?", "objection_timing"),
    (prospects[2], "I think our VP of Engineering would be the right person for this.", "objection_authority"),
]

for prospect, reply, obj_type in OBJECTION_DEMOS:
    followup = generate_followup(prospect, reply, obj_type)
    print(f"\n{'=' * 60}")
    print(f"Prospect  : {prospect['name']} \u2014 {prospect['contact_name']}")
    print(f"Objection : [{obj_type}] \"{reply}\"")
    print(f"Strategy  : {OBJECTION_STRATEGIES[obj_type][:80]}...")
    print(f"\nFollow-up subject: {followup.get('subject', '')}")
    print(followup.get("body", ""))

## Section 6 — End-to-end pipeline

Time to wire everything together. For each prospect we run:

```
Prospect → Research → Score → Email → Simulate Response → Classify → Follow-up
```

We add a simple lead-scoring step that uses the research brief's
confidence and number of pain signals to decide whether the prospect
is *hot*, *warm*, or *cold*. Only hot and warm prospects receive
outreach.

In [ ]:
SIMULATED_REPLIES = [
    "Sounds great, let's set up a call next week.",
    "Way too expensive for us at this stage.",
    "We're locked into a contract until Q4, circle back then.",
    "I'm not the right person — try emailing our CTO.",
    "Sure, send me some case studies and we'll talk.",
]

def score_lead(brief: dict, prospect: dict) -> dict:
    """Simple lead scorer using research-brief quality signals."""
    hooks = len(brief.get("personalization_hooks", []))
    pains = len(brief.get("pain_signals", []))
    confidence = brief.get("confidence", 0.5)
    score = round(0.3 * min(hooks / 5, 1) + 0.4 * min(pains / 4, 1) + 0.3 * confidence, 2)
    tier = "hot" if score >= 0.7 else ("warm" if score >= 0.4 else "cold")
    return {"score": score, "tier": tier}

# Run 5 prospects through the full pipeline
for idx, p in enumerate(prospects[:5]):
    reply_text = SIMULATED_REPLIES[idx % len(SIMULATED_REPLIES)]

    print(f"\n{'#' * 64}")
    print(f"# PROSPECT {idx + 1}: {p['name']} \u2014 {p['contact_name']} ({p['contact_title']})")
    print(f"{'#' * 64}")

    # Stage 1 — Research
    brief = research_prospect(p)
    print(f"\n[Research] Hooks found: {len(brief.get('personalization_hooks', []))}")
    for h in brief.get("personalization_hooks", []):
        print(f"  \u2022 {h}")

    # Stage 2 — Score
    lead = score_lead(brief, p)
    print(f"\n[Score] {lead['score']:.2f} \u2192 {lead['tier'].upper()}")

    if lead["tier"] == "cold":
        print("[Pipeline] Skipping cold lead.\n")
        continue

    # Stage 3 — Email
    email = generate_email(p, brief, ICP)
    print(f"\n[Email] Subject: {email.get('subject', 'N/A')}")
    print(email.get("body", ""))

    # Stage 4 — Simulated response
    print(f'\n[Response] "{reply_text}"')

    # Stage 5 — Classify
    classification = classify_response(reply_text)
    cat = classification.get("category", "unknown")
    print(f"[Classify] {cat} (confidence {classification.get('confidence', 0):.2f})")

    # Stage 6 — Follow-up (if objection)
    if cat.startswith("objection"):
        followup = generate_followup(p, reply_text, cat)
        print(f"\n[Follow-up] Subject: {followup.get('subject', '')}")
        print(followup.get("body", ""))
    elif cat == "positive":
        print("[Action] \u2192 Book meeting")
    else:
        print(f"[Action] \u2192 {cat} handling")

## Exercises

1. **Add a lead-scoring model with configurable weights** — Replace the hard-coded weights in `score_lead` with a `ScoringConfig` dataclass. Add an `engagement_score` dimension that estimates how actively the prospect is investing in their tech stack (based on job postings and funding stage). Score all 20 prospects and rank them.

2. **Multi-touch email sequence** — Extend `generate_email` to accept a `sequence_step` parameter (1, 2, or 3). Step 1 is the initial outreach. Step 2 is a follow-up that references the first email. Step 3 is a break-up email. Generate and print a full 3-step sequence for one prospect.

3. **Parallel research with batching** — Refactor `research_prospect` to process prospects in parallel using `asyncio` and the OpenAI async client. Benchmark the wall-clock time for researching all 20 prospects sequentially vs. in batches of 5.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A synthetic prospect database lets you develop and test the pipeline without live data. |
| 2 | The research agent is the highest-leverage component — email quality is bounded by research quality. |
| 3 | Side-by-side comparison makes the gap between generic and AI-personalized emails undeniable. |
| 4 | Few-shot classification with structured output reliably handles 7+ response categories. |
| 5 | Objection handling must be strategy-aware — a price objection and a timing objection need different follow-ups. |
| 6 | An end-to-end pipeline reveals integration bugs that unit-testing individual components never catches. |

## What's Next

- **Next**: [03_evaluate.ipynb](./03_evaluate.ipynb) — Evaluate email quality, classification accuracy, and pipeline ROI
- **Related**: [01_architecture.ipynb](./01_architecture.ipynb) — Revisit the architecture contracts if you need to modify data flows

## References & Further Reading

1. [OpenAI, "Structured Outputs," 2024](https://platform.openai.com/docs/guides/structured-outputs) — Best practices for reliable JSON generation from LLMs
2. [Salesloft, "Cadence Best Practices," 2024](https://www.salesloft.com/resources/cadence-best-practices) — Multi-touch sequence design patterns used by top SDR teams
3. [Apollo.io, "The State of Outbound in 2024"](https://www.apollo.io/blog/state-of-outbound) — Industry benchmarks for email personalization and response rates
4. [Gong Labs, "Cold Email Response Rates," 2023](https://www.gong.io/blog/cold-email-response-rates/) — Data on how personalization depth affects reply rates